In [ ]:
spark.stop()

In [1]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from sedona.spark import SedonaContext

# Configure Spark settings including driver memory and Azure authentication options.
conf = SparkConf() \
    .set("spark.driver.memory", "120g") \
    .set("spark.driver.maxResultSize", "4g") \
    .set("spark.sql.shuffle.partitions", "200") \
    .set("spark.sql.files.maxPartitionBytes", "256m") \
    .set("fs.azure.account.auth.type.propheusdatabay.dfs.core.windows.net", "OAuth") \
    .set("fs.azure.account.oauth.provider.type.propheusdatabay.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.MsiTokenProvider") \
    .set("fs.azure.account.oauth2.msi.tenant", "3f50e9d5-8877-43f0-bc25-fcecafe59ee5") \
    .set("fs.azure.account.oauth2.client.id", "322c41c6-38f5-4894-82a1-e955df89ff85") \
    .set("fs.azure.account.oauth2.client.endpoint", "http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=https://storage.azure.com/") \
    .set("fs.azure.account.oauth2.msi.endpoint", "http://169.254.169.254/metadata/identity/oauth2/token") \
    .set("fs.azure.account.oauth2.use.metadata.header", "true") \
    .set("sedona.global.index", "true") \
    .set("sedona.global.indextype", "rtree")
    

# Create Spark Context
sc = SparkContext(conf=conf)

# Create SparkSession
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .appName("SedonaApp") \
    .getOrCreate()

# Initialize Sedona Context (this registers Sedona's SQL functions and spatial types)
sedona = SedonaContext.create(spark)

# Confirm by printing Spark and Sedona contexts
print("Spark Session and SedonaContext have been successfully initiated.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/29 10:26:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/29 10:26:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/29 10:26:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/29 10:26:53 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark Session and SedonaContext have been successfully initiated.


In [5]:
path = "abfss://propheus-data-staging@propheusdatabay.dfs.core.windows.net/propheus-veraset-data/movement_eval/2025/11/"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(path)
)


In [6]:
df.printSchema()

root
 |-- ad_id: string (nullable = true)
 |-- utc_timestamp: timestamp (nullable = true)
 |-- horizontal_accuracy: string (nullable = true)
 |-- id_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- iso_country_code: string (nullable = true)
 |-- quality_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- geo_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [7]:
from pyspark.sql.functions import *

df.filter(col('longitude').isNull()).count()

0

In [8]:
df = df.withColumn("timestamp_th", from_utc_timestamp(col('utc_timestamp'), 'Asia/Bangkok')) \
    .withColumn("day", to_date(col("timestamp_th"))) \
    .withColumn("Ping_geom", expr("ST_Point(longitude, latitude)"))
df.show()

+--------------------+--------------------+-------------------+-------+---------------+---------+----------+----------------+--------------------+--------------------+--------------------+----------+--------------------+
|               ad_id|       utc_timestamp|horizontal_accuracy|id_type|     ip_address| latitude| longitude|iso_country_code|      quality_fields|          geo_fields|        timestamp_th|       day|           Ping_geom|
+--------------------+--------------------+-------------------+-------+---------------+---------+----------+----------------+--------------------+--------------------+--------------------+----------+--------------------+
|81f2e4c0-c7e2-451...| 2025-11-26 05:05:44|               66.0|   idfa|           NULL|14.062118|100.609122|              TH|{ping_sink_matche...|{region -> pathum...| 2025-11-26 12:05:44|2025-11-26|POINT (100.609122...|
|27a21411-00c6-4cd...|2025-11-26 16:13:...|                0.0|   idfa|110.168.246.148|   7.1966|  100.5873|        

In [9]:
df.printSchema()

root
 |-- ad_id: string (nullable = true)
 |-- utc_timestamp: timestamp (nullable = true)
 |-- horizontal_accuracy: string (nullable = true)
 |-- id_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- iso_country_code: string (nullable = true)
 |-- quality_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- geo_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- timestamp_th: timestamp (nullable = true)
 |-- day: date (nullable = true)
 |-- Ping_geom: geometry (nullable = true)



In [10]:
import geopandas as gpd

adm3_df = spark.read.parquet('admin3_geom')

adm3_df.count()

7425

In [11]:
adm3_df.printSchema()

root
 |-- adm3_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- geometry: geometry (nullable = true)



In [13]:
import logging

LOG_FILE = "mobility_w_subdistricts_write.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


logger = logging.getLogger("mobility_logger")
logger.setLevel(logging.INFO)


DEST_PATH = (
    "abfss://propheus-data-staging@propheusdatabay.dfs.core.windows.net/"
    "propheus-veraset-data/movement_eval_w_subdistricts/2025/11/"
)

if not logger.handlers:
    file_handler = logging.FileHandler(LOG_FILE)
    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(message)s"
    )
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)



In [ ]:

try:
    logging.info("Starting DAY-WISE mobility_w_subdistricts write job")

    days = df.select("day").distinct().orderBy("day").collect()

    logging.info(f"Found {len(days)} distinct days to process")

    for d in days:
        day_val = d["day"]

        logging.info(f"Starting processing for day={day_val}")

        df_day = df.filter(col("day") == day_val)
        logging.info(f"Found {df_day.count()} pings on day {d}")

        df_w_subdistricts = (
            df.alias("a")
            .join(
                adm3_df.alias("b"),
                expr("ST_Contains(b.geometry, a.Ping_geom)"),
                "inner"
            )
            .select(
                "a.*",
                col("b.adm1_name"),
                col("b.adm2_name"),
                col("b.adm3_name")
            )
        )

        day_path = f"{DEST_PATH}/day={day_val}"

        df_w_subdistricts.write \
            .mode("overwrite") \
            .parquet(day_path)

        logging.info(f"Successfully wrote data for day={day_val} to {day_path}")

    logging.info("Completed DAY-WISE mobility_w_subdistricts write job")

except Exception as e:
    logging.error(
        "DAY-WISE mobility_w_subdistricts job failed",
        exc_info=True
    )
    raise


[Stage 20:=====>       (384 + 48) / 854][Stage 21:>               (0 + 0) / 854]